# FINVERSE Ontology A2A Agent

Anthropic Claude를 사용하는 Moderator Agent와 네 개의 Domain Agent를 순서대로 테스트한다.

실행 순서:
1. 패키지·환경 설정
2. 함수와 PostgreSQL 도구 정의
3. 함수별 단위 테스트
4. 도메인별 DB 조회 테스트
5. Subagent 구성 테스트
6. 마지막 셀에서 실제 Agent 실행

In [ ]:
import sys
import subprocess

print("Python:", sys.version)
print("Interpreter:", sys.executable)
if sys.version_info < (3, 11):
    raise RuntimeError("deepagents requires Python 3.11 or newer. Change this Jupyter kernel.")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U",
    "--index-url", "https://pypi.org/simple",
    "deepagents", "langchain-anthropic", "psycopg[binary]",
])

In [ ]:
import getpass
import json
import os
import re
from pathlib import Path
from typing import Any

import psycopg
from psycopg.rows import dict_row
from deepagents import create_deep_agent
from langchain.tools import tool
from langchain_anthropic import ChatAnthropic

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "output" / "kospi-after-today"
QUERY = "오늘 이후 코스피가 어떻게 변할까?"
MODEL_NAME = "claude-sonnet-4-6"

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

if not os.environ.get("DATABASE_URL"):
    os.environ["DATABASE_URL"] = getpass.getpass("PostgreSQL DATABASE_URL: ")

print("환경 설정 완료")
print(f"model={MODEL_NAME}")
print(f"output={OUTPUT_DIR}")

## 1. 공통 설정과 함수

In [ ]:
DOMAIN_FILES = {
    "market": "market-evidence.md",
    "economy": "economic-evidence.md",
    "events": "external-event-evidence.md",
    "psychology": "psychology-evidence.md",
}

DOMAIN_VIEWS = {
    "market": (
        "market.index_daily",
        "market.price_daily",
        "market.investor_flow_daily",
        "market.foreign_holding_daily",
    ),
    "economy": ("economy.observation", "economy.series"),
    "events": ("events.news", "events.news_daily"),
    "psychology": ("psychology.sentiment_daily", "psychology.narratives"),
}

def slug(value: str) -> str:
    """Create a safe folder name from a user query."""
    value = re.sub(r"[^\w가-힣]+", "-", value, flags=re.UNICODE)
    return value.strip("-").lower()[:80] or "ontology-run"

def check(name: str, condition: bool, detail: str = "") -> None:
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {name}{(': ' + detail) if detail else ''}")

def query_domain(domain: str, limit: int = 10) -> dict[str, Any]:
    """Read bounded rows from the approved PostgreSQL views for one domain."""
    if domain not in DOMAIN_VIEWS:
        raise ValueError(f"Unknown domain: {domain}")
    safe_limit = max(1, min(int(limit), 100))
    result: dict[str, Any] = {"domain": domain, "views": {}}
    with psycopg.connect(os.environ["DATABASE_URL"], row_factory=dict_row) as conn:
        for view in DOMAIN_VIEWS[domain]:
            try:
                with conn.cursor() as cur:
                    cur.execute(f"SELECT * FROM {view} LIMIT %s", (safe_limit,))
                    result["views"][view] = cur.fetchall()
            except psycopg.Error as exc:
                conn.rollback()
                result["views"][view] = {"unavailable": str(exc).splitlines()[0]}
    return result

def save_evidence(domain: str, markdown: str, output_dir: Path = OUTPUT_DIR) -> Path:
    """Save one domain's Evidence Markdown file."""
    if domain not in DOMAIN_FILES:
        raise ValueError(f"Unknown domain: {domain}")
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / DOMAIN_FILES[domain]
    path.write_text(markdown.strip() + "\n", encoding="utf-8")
    return path

def read_evidence(output_dir: Path = OUTPUT_DIR) -> dict[str, str]:
    """Read all four Evidence Markdown files."""
    return {
        domain: (output_dir / filename).read_text(encoding="utf-8")
        if (output_dir / filename).exists() else "MISSING"
        for domain, filename in DOMAIN_FILES.items()
    }

## 2. 함수 테스트

In [ ]:
check("slug", slug(QUERY) == "오늘-이후-코스피가-어떻게-변할까")
check("domain configuration", set(DOMAIN_FILES) == {"market", "economy", "events", "psychology"})

test_dir = OUTPUT_DIR / "_function_test"
test_path = save_evidence("market", "# Market Evidence\n\n## Test\n- ok", test_dir)
loaded = read_evidence(test_dir)
check("save_evidence", test_path.exists(), str(test_path))
check("read_evidence", loaded["market"].startswith("# Market Evidence"))

try:
    query_domain("unknown")
except ValueError:
    check("invalid domain guard", True)
else:
    check("invalid domain guard", False)

## 3. PostgreSQL 연결 테스트

In [ ]:
try:
    with psycopg.connect(os.environ["DATABASE_URL"]) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            check("PostgreSQL connection", cur.fetchone()[0] == 1)
except Exception as exc:
    check("PostgreSQL connection", False, str(exc).splitlines()[0])
    print("DB가 연결되지 않아 다음 도메인 조회도 unavailable로 표시될 수 있습니다.")

## 4. 도메인별 데이터 조회 테스트

각 도메인을 하나씩 조회한다. 심리 view처럼 아직 DB에 없는 view는 실패하지 않고 `unavailable`로 기록한다.

In [ ]:
domain_samples = {}
for domain in DOMAIN_FILES:
    print(f"\n===== {domain} =====")
    try:
        domain_samples[domain] = query_domain(domain, limit=3)
        print(json.dumps(domain_samples[domain], ensure_ascii=False, indent=2, default=str)[:3000])
        check(f"query {domain}", "views" in domain_samples[domain])
    except Exception as exc:
        check(f"query {domain}", False, str(exc).splitlines()[0])

## 5. LangChain 도구와 Subagent 정의

In [ ]:
def make_query_tool(domain: str):
    @tool(f"query_{domain}_data")
    def query_tool(limit: int = 25) -> str:
        """Query approved PostgreSQL views for this domain."""
        try:
            return json.dumps(query_domain(domain, limit), ensure_ascii=False, default=str)
        except Exception as exc:
            return json.dumps({"domain": domain, "error": str(exc)})
    return query_tool

def make_save_tool(domain: str, output_dir: Path = OUTPUT_DIR):
    @tool(f"save_{domain}_evidence")
    def save_tool(markdown: str) -> str:
        """Save the completed Evidence Markdown for this domain."""
        return str(save_evidence(domain, markdown, output_dir))
    return save_tool

def make_subagent(domain: str, output_dir: Path = OUTPUT_DIR) -> dict[str, Any]:
    labels = {
        "market": "시장",
        "economy": "경제",
        "events": "외부 사건",
        "psychology": "사람들의 심리",
    }
    label = labels[domain]
    return {
        "name": f"{domain}-agent",
        "description": f"PostgreSQL evidence collector for {label}.",
        "tools": [make_query_tool(domain), make_save_tool(domain, output_dir)],
        "system_prompt": f"""
너는 FINVERSE의 {label} Agent다.
사용자 질문에 대한 최종 예측을 하지 말고 승인된 PostgreSQL view만 조회한다.
query_{domain}_data를 먼저 사용하고, 사실·해석·후보 관계·불확실성·부족한 데이터를 구분한다.
다음 Markdown 구조로 작성한 뒤 save_{domain}_evidence를 호출한다.
# {label} Evidence
## Current State
## Main Factors
## Relation Candidates
## Uncertainties
## Limitations
데이터가 없으면 추측하지 말고 Limitations에 기록한다.
""".strip(),
    }

def build_moderator(output_dir: Path = OUTPUT_DIR):
    @tool
    def read_evidence_documents() -> str:
        """Read all Evidence Markdown files for moderator review."""
        return json.dumps(read_evidence(output_dir), ensure_ascii=False)

    model = ChatAnthropic(model=MODEL_NAME, max_tokens=4096)
    return create_deep_agent(
        model=model,
        tools=[read_evidence_documents],
        system_prompt="""
너는 FINVERSE Moderator Agent다.
사용자 질문을 네 개의 Domain Agent에게 위임한다.
각 Agent의 Markdown 파일을 읽고 기준 시각, 출처, 사실과 해석의 분리, 부족한 데이터를 검토한다.
부족한 내용이 있으면 같은 Agent에 한 번 보완 요청한다.
최종 시장 예측이나 투자 추천은 하지 않는다.
""".strip(),
        subagents=[make_subagent(domain, output_dir) for domain in DOMAIN_FILES],
    )

check("query tool", make_query_tool("market").name == "query_market_data")
check("save tool", make_save_tool("market").name == "save_market_evidence")
subagent_names = [make_subagent(domain)["name"] for domain in DOMAIN_FILES]
check("four subagents", len(subagent_names) == 4, str(subagent_names))
print("Moderator 생성 테스트는 다음 셀에서 API 호출과 함께 실행합니다.")

## 6. 실제 A2A 실행

아래 셀부터 Anthropic API를 호출한다. 실행하면 네 개의 Evidence Markdown 파일이 생성된다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
moderator = build_moderator(OUTPUT_DIR)
result = moderator.invoke({"messages": [{"role": "user", "content": QUERY}]})
print(result["messages"][-1].content)

In [ ]:
from IPython.display import Markdown, display

for domain, filename in DOMAIN_FILES.items():
    path = OUTPUT_DIR / filename
    print(f"\n===== {domain}: {path} =====")
    if path.exists():
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print("MISSING")